# Analisi dei risultati della simulazione phishing

Questo notebook analizza i CSV generati da `simulations/run_simulation.py`.

L'obiettivo non è stimare il comportamento reale della popolazione, ma confrontare in modo controllato come agenti sintetici con profili diversi reagiscono a scenari di phishing e messaggi legittimi.

Il notebook distingue tra:

- **click / interazione iniziale**: l'agente apre un link o mostra interesse;
- **compromissione stretta**: l'agente inserisce credenziali o seed phrase;
- **segnalazione**: l'agente riconosce il messaggio come sospetto;
- **falso positivo**: l'agente segnala come phishing un messaggio legittimo.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path("results")
PLOTS_DIR = RESULTS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)


## 1. Caricamento del dataset

Il codice carica automaticamente il CSV più recente nella cartella `results/`.

Se vuoi analizzare un file specifico, sostituisci `csv_path` con il percorso desiderato.


In [ ]:
csv_files = sorted(RESULTS_DIR.glob("sim_*.csv"))

if not csv_files:
    raise FileNotFoundError(
        "Nessun file sim_*.csv trovato in results/. "
        "Esegui prima: python simulations/run_simulation.py"
    )

csv_path = csv_files[-1]
print(f"File caricato: {csv_path}")

df = pd.read_csv(csv_path)
df.head()


## 2. Normalizzazione e controlli preliminari

Questa sezione:

- rende il notebook compatibile con CSV generati da versioni diverse dello script;
- normalizza eventuali varianti testuali delle scelte prodotte dal modello;
- distingue i messaggi di phishing dai messaggi legittimi;
- separa gli errori tecnici (`PARSE_ERROR`) dalle decisioni valide degli agenti;
- identifica gli scenari ispirati al caso studio, quando presenti.


In [ ]:
CHOICE_MAPPING = {
    "1": "IGNORA",
    "1) IGNORA": "IGNORA",

    "2": "APRE_LINK",
    "2) APRE_LINK": "APRE_LINK",
    "CLICCA_SUL_LINK": "APRE_LINK",
    "CLICK_SUL_LINK": "APRE_LINK",
    "CLICKA_SUL_LINK": "APRE_LINK",

    "3": "COLLEGA_WALLET_O_APPROVA_TRANSAZIONE",
    "3) COLLEGA_WALLET_O_APPROVA_TRANSAZIONE": "COLLEGA_WALLET_O_APPROVA_TRANSAZIONE",
    "COLLEGA_WALLET": "COLLEGA_WALLET_O_APPROVA_TRANSAZIONE",
    "APPROVA_TRANSAZIONE": "COLLEGA_WALLET_O_APPROVA_TRANSAZIONE",

    "4": "INSERISCE_CREDENZIALI_O_SEED",
    "4) INSERISCE_CREDENZIALI_O_SEED": "INSERISCE_CREDENZIALI_O_SEED",
    "INSERISCI_CREDENZIALI": "INSERISCE_CREDENZIALI_O_SEED",
    "INSERISCE_CREDENZIALI": "INSERISCE_CREDENZIALI_O_SEED",
    "INSERISCE_SEED": "INSERISCE_CREDENZIALI_O_SEED",
    "FORNISCE_SEED": "INSERISCE_CREDENZIALI_O_SEED",

    "5": "CONCEDE_ACCESSO_REMOTO",
    "5) CONCEDE_ACCESSO_REMOTO": "CONCEDE_ACCESSO_REMOTO",
    "INSTALLA_SOFTWARE_ACCESSO_REMOTO": "CONCEDE_ACCESSO_REMOTO",
    "CONCEDE_CONTROLLO_REMOTO": "CONCEDE_ACCESSO_REMOTO",
    "CONDIVIDE_SCHERMO": "CONCEDE_ACCESSO_REMOTO",

    "6": "INVIA_FONDI",
    "6) INVIA_FONDI": "INVIA_FONDI",

    "7": "VERIFICA_TRAMITE_CANALE_UFFICIALE",
    "7) VERIFICA_TRAMITE_CANALE_UFFICIALE": "VERIFICA_TRAMITE_CANALE_UFFICIALE",
    "CHIEDI_AIUTO_AMICO": "VERIFICA_TRAMITE_CANALE_UFFICIALE",
    "CHIEDE_AIUTO": "VERIFICA_TRAMITE_CANALE_UFFICIALE",

    "8": "SEGNALA_COME_PHISHING",
    "8) SEGNALA_COME_PHISHING": "SEGNALA_COME_PHISHING",
}

ALLOWED_CHOICES = {
    "IGNORA",
    "APRE_LINK",
    "COLLEGA_WALLET_O_APPROVA_TRANSAZIONE",
    "INSERISCE_CREDENZIALI_O_SEED",
    "CONCEDE_ACCESSO_REMOTO",
    "INVIA_FONDI",
    "VERIFICA_TRAMITE_CANALE_UFFICIALE",
    "SEGNALA_COME_PHISHING",
    "PARSE_ERROR",
}

if "raw_choice" not in df.columns:
    df["raw_choice"] = df["choice"]

if "parse_error" not in df.columns:
    fallback_mask = df.get("motivation", "").astype(str).str.contains(
        "fallback|errore parsing|errore di parsing", case=False, na=False
    )
    df["parse_error"] = fallback_mask

if "run_id" not in df.columns:
    df["run_id"] = csv_path.stem.replace("sim_", "")

if "model" not in df.columns:
    df["model"] = "unknown"

if "message_type" in df.columns:
    df["message_type"] = df["message_type"].replace({
        "legitimo": "legittimo",
        "legitimate": "legittimo",
        "legittima": "legittimo",
    })

normalized = (
    df["choice"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df["choice_normalized"] = normalized.map(lambda x: CHOICE_MAPPING.get(x, x))
df.loc[~df["choice_normalized"].isin(ALLOWED_CHOICES), "choice_normalized"] = "PARSE_ERROR"

if df["parse_error"].dtype == object:
    df["parse_error"] = df["parse_error"].astype(str).str.lower().isin(["true", "1", "yes"])

df.loc[df["choice_normalized"] == "PARSE_ERROR", "parse_error"] = True

# Identifica scenari ispirati al caso studio.
# Il CSV non salva ancora esplicitamente la feature "case_study", quindi usiamo il prefisso dell'ID.
df["is_case_study"] = df["message_id"].astype(str).str.startswith("case_study_")

valid_df = df[~df["parse_error"]].copy()
phishing_df = valid_df[valid_df["message_type"] == "phishing"].copy()
legit_df = valid_df[valid_df["message_type"] == "legittimo"].copy()

generic_phishing_df = phishing_df[~phishing_df["is_case_study"]].copy()
case_study_df = phishing_df[phishing_df["is_case_study"]].copy()

summary = pd.DataFrame({
    "metrica": [
        "righe totali",
        "righe valide",
        "parse error",
        "agenti unici",
        "messaggi unici",
        "messaggi phishing validi",
        "messaggi legittimi validi",
        "interazioni phishing generiche",
        "interazioni caso studio",
    ],
    "valore": [
        len(df),
        len(valid_df),
        int(df["parse_error"].sum()),
        df["agent_id"].nunique() if "agent_id" in df.columns else None,
        df["message_id"].nunique() if "message_id" in df.columns else None,
        len(phishing_df),
        len(legit_df),
        len(generic_phishing_df),
        len(case_study_df),
    ]
})

summary


## 3. Distribuzione globale delle scelte

Questa tabella serve a capire il comportamento complessivo degli agenti, ma non distingue ancora tra phishing e messaggi legittimi.


In [ ]:
choice_distribution = (
    valid_df["choice_normalized"]
    .value_counts()
    .rename_axis("choice")
    .reset_index(name="count")
)
choice_distribution["percentage"] = choice_distribution["count"] / choice_distribution["count"].sum() * 100
choice_distribution


In [ ]:
ax = choice_distribution.plot(
    kind="bar",
    x="choice",
    y="percentage",
    legend=False,
    figsize=(9, 5),
)
ax.set_title("Distribuzione globale delle scelte")
ax.set_xlabel("Scelta")
ax.set_ylabel("Percentuale")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "global_choice_distribution.png", dpi=200)
plt.show()


## 4. Metriche corrette per i messaggi di phishing

Per i messaggi di phishing, distinguiamo:

- **click rate**: l'agente apre il link;
- **compromissione stretta**: l'agente inserisce credenziali o seed phrase;
- **failure loose**: l'agente compie una qualunque azione rischiosa, includendo il click;
- **reporting rate**: l'agente segnala il messaggio come phishing.

Il click non equivale automaticamente a compromissione: indica solo una prima interazione rischiosa.


In [ ]:
if phishing_df.empty:
    raise ValueError("Nessun messaggio phishing valido trovato nel dataset.")

phishing_df["click_rate"] = phishing_df["choice_normalized"] == "APRE_LINK"

phishing_df["wallet_or_transaction_rate"] = phishing_df["choice_normalized"] == "COLLEGA_WALLET_O_APPROVA_TRANSAZIONE"
phishing_df["credential_seed_disclosure"] = phishing_df["choice_normalized"] == "INSERISCE_CREDENZIALI_O_SEED"
phishing_df["remote_access_rate"] = phishing_df["choice_normalized"] == "CONCEDE_ACCESSO_REMOTO"
phishing_df["fund_transfer_rate"] = phishing_df["choice_normalized"] == "INVIA_FONDI"

phishing_df["strict_compromise"] = phishing_df["choice_normalized"].isin([
    "COLLEGA_WALLET_O_APPROVA_TRANSAZIONE",
    "INSERISCE_CREDENZIALI_O_SEED",
    "CONCEDE_ACCESSO_REMOTO",
    "INVIA_FONDI",
])

phishing_df["failure_loose"] = phishing_df["choice_normalized"].isin([
    "APRE_LINK",
    "COLLEGA_WALLET_O_APPROVA_TRANSAZIONE",
    "INSERISCE_CREDENZIALI_O_SEED",
    "CONCEDE_ACCESSO_REMOTO",
    "INVIA_FONDI",
])

phishing_df["reporting_rate"] = phishing_df["choice_normalized"] == "SEGNALA_COME_PHISHING"
phishing_df["verification_rate"] = phishing_df["choice_normalized"] == "VERIFICA_TRAMITE_CANALE_UFFICIALE"
phishing_df["ignore_rate"] = phishing_df["choice_normalized"] == "IGNORA"

phishing_metrics = pd.DataFrame({
    "metrica": [
        "click rate",
        "wallet/transaction approval rate",
        "credential/seed disclosure rate",
        "remote access rate",
        "fund transfer rate",
        "strict compromise rate",
        "failure loose rate",
        "reporting rate",
        "verification rate",
        "ignore rate",
    ],
    "percentuale": [
        phishing_df["click_rate"].mean() * 100,
        phishing_df["wallet_or_transaction_rate"].mean() * 100,
        phishing_df["credential_seed_disclosure"].mean() * 100,
        phishing_df["remote_access_rate"].mean() * 100,
        phishing_df["fund_transfer_rate"].mean() * 100,
        phishing_df["strict_compromise"].mean() * 100,
        phishing_df["failure_loose"].mean() * 100,
        phishing_df["reporting_rate"].mean() * 100,
        phishing_df["verification_rate"].mean() * 100,
        phishing_df["ignore_rate"].mean() * 100,
    ]
})

phishing_metrics


## 5. Metriche per i messaggi legittimi

Per i messaggi legittimi non ha senso parlare di compromissione se l'agente clicca.

In questo caso misuriamo soprattutto:

- **legitimate click rate**: l'agente interagisce con un messaggio legittimo;
- **false positive rate**: l'agente segnala come phishing un messaggio legittimo.


In [ ]:
if legit_df.empty:
    print("Nessun messaggio legittimo valido trovato nel dataset.")
    legit_metrics = pd.DataFrame(columns=["metrica", "percentuale"])
else:
    legit_df["legitimate_click_rate"] = legit_df["choice_normalized"] == "CLICCA_SUL_LINK"
    legit_df["false_positive_rate"] = legit_df["choice_normalized"] == "SEGNALA_COME_PHISHING"
    legit_df["ignore_rate"] = legit_df["choice_normalized"] == "IGNORA"

    legit_metrics = pd.DataFrame({
        "metrica": [
            "legitimate click rate",
            "false positive rate",
            "ignore rate",
        ],
        "percentuale": [
            legit_df["legitimate_click_rate"].mean() * 100,
            legit_df["false_positive_rate"].mean() * 100,
            legit_df["ignore_rate"].mean() * 100,
        ]
    })

legit_metrics


## 6. Risultati per scenario

Questa tabella mostra quali scenari generano più interazioni rischiose e quali vengono riconosciuti più spesso come phishing.


In [ ]:
by_message = (
    phishing_df
    .groupby(["message_id", "scenario_description"], dropna=False)
    .agg(
        n=("choice_normalized", "size"),
        click_rate=("click_rate", "mean"),
        wallet_or_transaction_rate=("wallet_or_transaction_rate", "mean"),
        credential_seed_disclosure=("credential_seed_disclosure", "mean"),
        remote_access_rate=("remote_access_rate", "mean"),
        fund_transfer_rate=("fund_transfer_rate", "mean"),
        strict_compromise=("strict_compromise", "mean"),
        failure_loose=("failure_loose", "mean"),
        reporting_rate=("reporting_rate", "mean"),
        verification_rate=("verification_rate", "mean"),
        ignore_rate=("ignore_rate", "mean"),
    )
    .reset_index()
)

percentage_cols = [
    "click_rate",
    "wallet_or_transaction_rate",
    "credential_seed_disclosure",
    "remote_access_rate",
    "fund_transfer_rate",
    "strict_compromise",
    "failure_loose",
    "reporting_rate",
    "verification_rate",
    "ignore_rate",
]
by_message[percentage_cols] = by_message[percentage_cols] * 100
by_message = by_message.sort_values("failure_loose", ascending=False)
by_message


In [ ]:
plot_df = by_message.sort_values("failure_loose", ascending=True)
ax = plot_df.plot(
    kind="barh",
    x="message_id",
    y="failure_loose",
    legend=False,
    figsize=(9, 5),
)
ax.set_title("Failure loose per scenario phishing")
ax.set_xlabel("Percentuale")
ax.set_ylabel("Scenario")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "failure_loose_by_message.png", dpi=200)
plt.show()


## 7. Risultati per livello di formazione

Questa sezione confronta gli agenti in base alla formazione dichiarata nel profilo.


In [ ]:
by_training = (
    phishing_df
    .groupby("security_training", dropna=False)
    .agg(
        n=("choice_normalized", "size"),
        click_rate=("click_rate", "mean"),
        wallet_or_transaction_rate=("wallet_or_transaction_rate", "mean"),
        credential_seed_disclosure=("credential_seed_disclosure", "mean"),
        remote_access_rate=("remote_access_rate", "mean"),
        fund_transfer_rate=("fund_transfer_rate", "mean"),
        strict_compromise=("strict_compromise", "mean"),
        failure_loose=("failure_loose", "mean"),
        reporting_rate=("reporting_rate", "mean"),
        verification_rate=("verification_rate", "mean"),
        ignore_rate=("ignore_rate", "mean"),
    )
    .reset_index()
)
by_training[percentage_cols] = by_training[percentage_cols] * 100
by_training = by_training.sort_values("failure_loose", ascending=False)
by_training


In [ ]:
plot_df = by_training.sort_values("failure_loose", ascending=True)
ax = plot_df.plot(
    kind="barh",
    x="security_training",
    y="failure_loose",
    legend=False,
    figsize=(8, 4),
)
ax.set_title("Failure loose per livello di formazione")
ax.set_xlabel("Percentuale")
ax.set_ylabel("Formazione")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "failure_loose_by_training.png", dpi=200)
plt.show()


## 8. Risultati per fascia d'età

Questa analisi è esplorativa: le fasce d'età degli agenti non rappresentano un campione statistico della popolazione reale.


In [ ]:
by_age = (
    phishing_df
    .groupby("age_group", dropna=False)
    .agg(
        n=("choice_normalized", "size"),
        click_rate=("click_rate", "mean"),
        wallet_or_transaction_rate=("wallet_or_transaction_rate", "mean"),
        credential_seed_disclosure=("credential_seed_disclosure", "mean"),
        remote_access_rate=("remote_access_rate", "mean"),
        fund_transfer_rate=("fund_transfer_rate", "mean"),
        strict_compromise=("strict_compromise", "mean"),
        failure_loose=("failure_loose", "mean"),
        reporting_rate=("reporting_rate", "mean"),
        verification_rate=("verification_rate", "mean"),
        ignore_rate=("ignore_rate", "mean"),
    )
    .reset_index()
)
by_age[percentage_cols] = by_age[percentage_cols] * 100
by_age = by_age.sort_values("age_group")
by_age


In [ ]:
ax = by_age.plot(
    kind="bar",
    x="age_group",
    y="failure_loose",
    legend=False,
    figsize=(8, 4),
)
ax.set_title("Failure loose per fascia d'età")
ax.set_xlabel("Fascia d'età")
ax.set_ylabel("Percentuale")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "failure_loose_by_age_group.png", dpi=200)
plt.show()


## 9. Falsi positivi sui messaggi legittimi

Questa sezione serve a misurare se gli agenti tendono a essere eccessivamente sospettosi.


In [ ]:
if legit_df.empty:
    by_legit_message = pd.DataFrame()
else:
    by_legit_message = (
        legit_df
        .groupby(["message_id", "scenario_description"], dropna=False)
        .agg(
            n=("choice_normalized", "size"),
            legitimate_click_rate=("legitimate_click_rate", "mean"),
            false_positive_rate=("false_positive_rate", "mean"),
            ignore_rate=("ignore_rate", "mean"),
        )
        .reset_index()
    )
    by_legit_message[["legitimate_click_rate", "false_positive_rate", "ignore_rate"]] *= 100

by_legit_message


## 10. Focus sugli scenari ispirati al caso studio

Questa sezione separa gli scenari generici dagli scenari ispirati al caso studio del 18 agosto 2024. L'obiettivo non è ricostruire con certezza il comportamento della vittima reale, ma osservare come agenti sintetici e profili ad alto valore reagiscono a messaggi mirati basati su impersonificazione del supporto, urgenza e possibile accesso remoto.


In [ ]:
if case_study_df.empty:
    print("Nessuno scenario case_study_ trovato nel dataset.")
    case_study_metrics = pd.DataFrame()
else:
    case_study_metrics = (
        case_study_df
        .groupby(["message_id", "scenario_description"], dropna=False)
        .agg(
            n=("choice_normalized", "size"),
            click_rate=("click_rate", "mean"),
            remote_access_rate=("remote_access_rate", "mean"),
            strict_compromise=("strict_compromise", "mean"),
            failure_loose=("failure_loose", "mean"),
            reporting_rate=("reporting_rate", "mean"),
            verification_rate=("verification_rate", "mean"),
            ignore_rate=("ignore_rate", "mean"),
        )
        .reset_index()
    )
    case_cols = [
        "click_rate", "remote_access_rate", "strict_compromise",
        "failure_loose", "reporting_rate", "verification_rate", "ignore_rate"
    ]
    case_study_metrics[case_cols] = case_study_metrics[case_cols] * 100

case_study_metrics


In [ ]:
case_profile_df = phishing_df[
    phishing_df["agent_id"].astype(str).str.startswith("case_study_genesis_creditor_high_value_holder")
].copy()

if case_profile_df.empty:
    print("Profilo case_study_genesis_creditor_high_value_holder non trovato nei risultati.")
    case_profile_summary = pd.DataFrame()
else:
    case_profile_summary = (
        case_profile_df
        .groupby(["message_id", "scenario_description", "choice_normalized"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values(["message_id", "count"], ascending=[True, False])
    )

case_profile_summary


## 10. Esportazione tabelle

Le tabelle principali vengono salvate in `results/plots/` per poterle usare nella tesi.


In [ ]:
choice_distribution.to_csv(PLOTS_DIR / "table_choice_distribution.csv", index=False)
phishing_metrics.to_csv(PLOTS_DIR / "table_phishing_metrics.csv", index=False)
legit_metrics.to_csv(PLOTS_DIR / "table_legitimate_metrics.csv", index=False)
by_message.to_csv(PLOTS_DIR / "table_phishing_by_message.csv", index=False)
by_training.to_csv(PLOTS_DIR / "table_phishing_by_training.csv", index=False)
by_age.to_csv(PLOTS_DIR / "table_phishing_by_age_group.csv", index=False)

if not by_legit_message.empty:
    by_legit_message.to_csv(PLOTS_DIR / "table_legitimate_by_message.csv", index=False)

print(f"Tabelle e grafici salvati in: {PLOTS_DIR}")
if 'case_study_metrics' in globals() and not case_study_metrics.empty:
    case_study_metrics.to_csv(PLOTS_DIR / "table_case_study_metrics.csv", index=False)
if 'case_profile_summary' in globals() and not case_profile_summary.empty:
    case_profile_summary.to_csv(PLOTS_DIR / "table_case_profile_summary.csv", index=False)


## 12. Interpretazione metodologica

I risultati devono essere letti come output di una simulazione controllata, non come misurazioni statistiche su persone reali.

In particolare:

- `APRE_LINK` rappresenta una prima interazione rischiosa, ma non implica automaticamente furto di credenziali, approvazione di transazioni o perdita di fondi;
- la compromissione stretta è associata ad azioni più gravi, come collegare un wallet, approvare una transazione, concedere accesso remoto, inserire credenziali/seed phrase o inviare fondi;
- `CONCEDE_ACCESSO_REMOTO` è particolarmente rilevante per gli scenari ispirati al caso studio, perché rappresenta una forma di compromissione operativa del dispositivo;
- i messaggi legittimi vanno analizzati separatamente, perché interagire con un messaggio legittimo non equivale a una compromissione;
- i valori percentuali servono soprattutto a confrontare scenari e profili tra loro, non a stimare la vulnerabilità reale della popolazione.

Una formulazione adatta alla tesi può essere:

> I risultati ottenuti non devono essere interpretati come una stima statistica del comportamento umano reale. Gli agenti sintetici non costituiscono un campione rappresentativo di utenti, ma permettono di osservare in modo controllato come differenti profili e caratteristiche del messaggio possano influenzare la propensione all'interazione con contenuti fraudolenti. Il valore dell'esperimento è quindi principalmente comparativo ed esplorativo.
